# Hybrid: uwsplat geometry → unchanged RUSplatting (Colab A100)

**Our part:** SIFT SfM cameras + dense cloud v2 (`mvs`) from `noUpKey/splat_dense-sparse` (branch `dense-v2`).
**RUSplatting's part, unchanged:** upstream `theflash987/RUSplatting` at commit `33e2a62` — RGB-decoupled medium MLP, its rasterizer, losses, densification, `render.py`, `metrics.py`.

Two runs per scene, same images, same depth maps, same settings — only the starting cloud differs:
- `sift` — RUSplatting on our SfM with our SIFT points only (**control**)
- `mvs` — RUSplatting on our SfM with our dense cloud v2 (**hybrid**)

`mvs − sift` is what our dense points add inside RUSplatting. Do not compare with the old 24.62 directly: those runs used COLMAP's 1328×739 undistorted images, these use our 1280×720 ones.

Everything is saved to Drive after each run; re-running the notebook skips finished runs.

In [ ]:
# ---- settings ----
SCENES = ["Tokai", "Cormoran", "Isro", "Kwaj"]   # Tokai and Cormoran first: the scenes where uwsplat struggles
CLOUDS = ["sift", "mvs"]                          # control, hybrid  (add "fusion" for dense cloud v1)
ITERS = 20000                                     # paper setting = our upstream reproduction (24.62); RUSplatting's code default is 15000
SAVE_PLY = False                                  # copy trained Gaussians to Drive too (~0.2-0.7 GB per run)

DRIVE_DIR = "/content/drive/MyDrive/uwsplat_colab"
UWS_DRIVE = f"{DRIVE_DIR}/outputs_v2"             # uwsplat outputs of batches 1-3 (sfm, sparse, images, mvs)
HYB_DRIVE = f"{DRIVE_DIR}/hybrid_rus_{ITERS // 1000}k"

UWS_REPO, RUS_REPO = "/content/splat_dense-sparse", "/content/RUSplatting"
UWS_OUT, SCN_DIR, MODEL_DIR = "/content/uws_out", "/content/rus_scenes", "/content/rus_out"

import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"
cc = torch.cuda.get_device_capability()
ARCH = f"{cc[0]}.{cc[1]}"
print("torch", torch.__version__, "CUDA", torch.version.cuda, "arch", ARCH)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(HYB_DRIVE, exist_ok=True)
print("uwsplat outputs on Drive:", sorted(os.listdir(UWS_DRIVE)) if os.path.isdir(UWS_DRIVE) else "none (geometry will be recomputed)")

In [ ]:
%%bash -s "$UWS_REPO" "$RUS_REPO"
set -e
if [ -d "$1/.git" ]; then git -C "$1" fetch -q origin dense-v2 && git -C "$1" reset -q --hard origin/dense-v2
else git clone -q --branch dense-v2 https://github.com/noUpKey/splat_dense-sparse "$1"; fi
git -C "$1" log --oneline -1
if [ ! -d "$2/.git" ]; then git clone -q https://github.com/theflash987/RUSplatting "$2"; fi
git -C "$2" checkout -q 33e2a62 && git -C "$2" log --oneline -1

In [ ]:
%%bash -s "$RUS_REPO" "$ARCH"
set -e
pip -q install plyfile tqdm imageio imageio-ffmpeg lpips safetensors ninja "gsplat==1.5.3" transformers
cd "$1"
# build fix only (no behaviour change): newer GCC needs <cstdint> for uint32_t / uintptr_t in the rasterizer
H=submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h
grep -q "<cstdint>" $H || sed -i '0,/#include <iostream>/s//#include <iostream>\n#include <cstdint>/' $H
export TORCH_CUDA_ARCH_LIST="$2"
python -c "import diff_gaussian_rasterization, simple_knn" 2>/dev/null && echo "RUSplatting extensions already built" || \
  pip -q install --no-build-isolation ./submodules/diff-gaussian-rasterization ./submodules/simple-knn
python -c "import diff_gaussian_rasterization, simple_knn._C; print('RUSplatting extensions OK')"

## 1. Our geometry
Restores each scene's uwsplat output from Drive (`sfm/`, `sparse/0/`, `images/`, `mvs/`). Scenes without it are recomputed (stages sfm → dense → fusion → mvs, ~3–5 min per scene incl. the 2.3 GB DUSt3R download once) and saved back to Drive.

In [ ]:
import os, shutil, subprocess, sys
NEED = ["sfm/scene.json", "sparse/0/images.txt", "sparse/0/cameras.txt", "sparse/0/points3D.txt", "images", "mvs/fused.ply"]
KEEP = ["sfm", "sparse", "images", "dense", "fusion", "mvs"]
os.makedirs(UWS_OUT, exist_ok=True)

def complete(d):
    return all(os.path.exists(os.path.join(d, n)) for n in NEED)

todo = []
for s in SCENES:
    local, remote = f"{UWS_OUT}/{s}", f"{UWS_DRIVE}/{s}"
    if complete(local):
        print(s, ": local copy ok")
    elif complete(remote):
        for k in KEEP:
            if os.path.exists(f"{remote}/{k}") and not os.path.exists(f"{local}/{k}"):
                shutil.copytree(f"{remote}/{k}", f"{local}/{k}")
        print(s, ": restored from Drive")
    else:
        todo.append(s)
if todo:
    print("recomputing geometry for", todo)
    subprocess.run([sys.executable, "scripts/download_weights.py"], cwd=UWS_REPO, check=True)
    for s in todo:
        subprocess.run([sys.executable, "scripts/run_scene.py", "--scene", f"Submerged3D/{s}", "--out", f"{UWS_OUT}/{s}",
                        "--stages", "sfm,dense,fusion,mvs"], cwd=UWS_REPO, check=True)
        for k in KEEP:
            if os.path.exists(f"{UWS_OUT}/{s}/{k}"):
                shutil.copytree(f"{UWS_OUT}/{s}/{k}", f"{UWS_DRIVE}/{s}/{k}", dirs_exist_ok=True)
for s in SCENES:
    assert complete(f"{UWS_OUT}/{s}"), f"{s}: geometry incomplete"
print("geometry ready for", SCENES)

## 2. RUSplatting scene folders + depth maps
`export_rusplatting.py` writes our images, PINHOLE cameras/poses and the chosen cloud as `sparse/0/points3D.ply`, which RUSplatting loads instead of COLMAP's points.
Depth maps: Depth-Anything-V2 Large, min–max to 8 bit, flipped `255 − d` (far = bright, matching RUSplatting's min–max-normalised rendered depth) — the same convention as the A40 runs. Computed once per scene and shared by both clouds.

In [ ]:
import glob, numpy as np, torch.nn.functional as F
from PIL import Image
from transformers import pipeline as hf_pipeline

for s in SCENES:
    for c in CLOUDS:
        dst = f"{SCN_DIR}/{s}_{c}"
        if not os.path.exists(f"{dst}/sparse/0/points3D.ply"):
            subprocess.run([sys.executable, "scripts/export_rusplatting.py", "--scene_out", f"{UWS_OUT}/{s}", "--dst", dst,
                            "--cloud", c, "--copy", "--force"], cwd=UWS_REPO, check=True)

da = None
for s in SCENES:
    cache = f"{UWS_OUT}/{s}/depthmap_da2l"
    imgs = sorted(glob.glob(f"{UWS_OUT}/{s}/images/*.png"))
    if not (os.path.isdir(cache) and len(os.listdir(cache)) == len(imgs)):
        if da is None:
            da = hf_pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Large-hf", device=0)
        os.makedirs(cache, exist_ok=True)
        for p in imgs:
            im = Image.open(p).convert("RGB")
            pd = da(im)["predicted_depth"].float()          # relative inverse depth (near = large)
            pd = F.interpolate(pd.reshape(1, 1, *pd.shape[-2:]), size=(im.height, im.width), mode="bicubic", align_corners=False)[0, 0]
            d = (pd - pd.min()) / (pd.max() - pd.min()).clamp_min(1e-8) * 255.0
            Image.fromarray((255 - d.clamp(0, 255).round()).byte().cpu().numpy(), "L").save(f"{cache}/{os.path.basename(p)}")
        print(s, ": depth maps", len(imgs))
    for c in CLOUDS:
        dst = f"{SCN_DIR}/{s}_{c}/depthmap"
        if not os.path.isdir(dst):
            shutil.copytree(cache, dst)
del da
torch.cuda.empty_cache()

## 3. Train, render, score (upstream `train.py` / `render.py` / `metrics.py`)
Each run: `train.py --eval --iterations ITERS`, `render.py --skip_train`, `metrics.py`. Log, `results.json`, `per_view.json`, test renders and config go to Drive. Time per run on an A100 is not measured yet; expect roughly 20–40 min.

In [ ]:
import json, time

def run(cmd, log):
    with open(log, "a") as f:
        f.write("\n$ " + " ".join(cmd) + "\n"); f.flush()
        r = subprocess.run(cmd, cwd=RUS_REPO, stdout=f, stderr=subprocess.STDOUT)
    if r.returncode:
        print(open(log).read()[-3000:])
        raise RuntimeError(f"failed: {' '.join(cmd[:2])}")

def save_to_drive(m, dst):
    for root, dirs, files in os.walk(m):
        rel = os.path.relpath(root, m)
        if not SAVE_PLY and rel.startswith("point_cloud"):
            continue
        for fn in files:
            os.makedirs(os.path.join(dst, rel), exist_ok=True)
            shutil.copy2(os.path.join(root, fn), os.path.join(dst, rel, fn))

t_all = time.time()
for s in SCENES:
    for c in CLOUDS:
        tag, m, drv = f"{s}_{c}", f"{MODEL_DIR}/{s}_{c}", f"{HYB_DRIVE}/{s}_{c}"
        if os.path.exists(f"{drv}/results.json"):
            print(tag, ": done earlier (Drive)"); continue
        shutil.rmtree(m, ignore_errors=True); os.makedirs(m)
        log, t0 = f"{m}/run.log", time.time()
        print(f"{tag}: training {ITERS} it. ...", flush=True)
        run([sys.executable, "train.py", "-s", f"{SCN_DIR}/{tag}", "-m", m, "--eval", "--iterations", str(ITERS),
             "--test_iterations", str(ITERS), "--save_iterations", str(ITERS), "--quiet"], log)
        run([sys.executable, "render.py", "-m", m, "--skip_train", "--quiet"], log)
        run([sys.executable, "metrics.py", "-m", m], log)
        r = json.load(open(f"{m}/results.json"))[f"ours_{ITERS}"]
        flag = "   <-- looks broken (NaN run? upstream has no NaN guard)" if r["PSNR"] < 12 else ""
        print(f"{tag}: PSNR {r['PSNR']:.2f}  SSIM {r['SSIM']:.4f}  LPIPS {r['LPIPS']:.4f}  ({(time.time() - t0) / 60:.1f} min){flag}", flush=True)
        save_to_drive(m, drv)
        shutil.copy2(f"{SCN_DIR}/{tag}/test_list.txt", drv)
print(f"all runs finished, {(time.time() - t_all) / 60:.0f} min this session")

## 4. Results

In [ ]:
import json, os
import numpy as np
REF = {  # context only: different images (COLMAP 1328x739) or different model
    "RUSplatting on COLMAP, upstream 20k (A40 repro)": {"Cormoran": (21.57, .690, .334), "Isro": (24.52, .870, .251), "Kwaj": (28.04, .886, .193), "Tokai": (24.37, .734, .354)},
    "uwsplat mvs_lpips (Colab A100, batch 2)": {"Tokai": (22.66, .6793, .4273), "Cormoran": (19.75, .6405, .4023), "Isro": (26.08, .8520, .3102), "Kwaj": (26.35, .8646, .2354)},
}

def res(s, c):
    p = f"{HYB_DRIVE}/{s}_{c}/results.json"
    if not os.path.exists(p):
        return None
    r = json.load(open(p))[f"ours_{ITERS}"]
    return r["PSNR"], r["SSIM"], r["LPIPS"]

f3 = lambda t: "–" if t is None else f"{t[0]:.2f} / {t[1]:.4f} / {t[2]:.4f}"
lines = ["| scene | " + " | ".join(f"RUS + our {c}" for c in CLOUDS) + " | mvs − sift |", "|---" * (len(CLOUDS) + 2) + "|"]
acc = {c: [] for c in CLOUDS}
for s in SCENES:
    row = {c: res(s, c) for c in CLOUDS}
    for c in CLOUDS:
        if row[c]: acc[c].append(row[c])
    d = "–"
    if row.get("sift") and row.get("mvs"):
        d = " / ".join(f"{a - b:+.{k}f}" for a, b, k in zip(row["mvs"], row["sift"], (2, 4, 4)))
    lines.append(f"| {s} | " + " | ".join(f3(row[c]) for c in CLOUDS) + f" | {d} |")
if all(len(acc[c]) == len(SCENES) for c in CLOUDS):
    mean = {c: tuple(np.mean(acc[c], 0)) for c in CLOUDS}
    d = " / ".join(f"{a - b:+.{k}f}" for a, b, k in zip(mean["mvs"], mean["sift"], (2, 4, 4))) if "mvs" in mean and "sift" in mean else "–"
    lines.append("| **mean** | " + " | ".join(f"**{f3(mean[c])}**" for c in CLOUDS) + f" | **{d}** |")
print("\n".join(lines))
print("\nReference (not directly comparable):")
for k, v in REF.items():
    print(f"  {k}: " + ", ".join(f"{s} {v[s][0]:.2f}" for s in SCENES) + f"  | mean {np.mean([v[s][0] for s in v]):.2f}")
print("\nPer test view (PSNR):")
for s in SCENES:
    for c in CLOUDS:
        p = f"{HYB_DRIVE}/{s}_{c}/per_view.json"
        if os.path.exists(p):
            pv = json.load(open(p))[f"ours_{ITERS}"]["PSNR"]
            print(f"  {s}_{c}: " + ", ".join(f"{k} {v:.2f}" for k, v in sorted(pv.items())))
open(f"{HYB_DRIVE}/results.md", "w").write("\n".join(lines) + "\n")